# 12 · Explanations for trajectory and risk (P6)

SHAP reasons per graft (exact for the Cox model), descriptive counterfactuals and sanity checks. Code: `explain/followup_explain.py`. Same as `python -m recommendation.main followup-explain`.

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
import joblib
from recommendation.data_prep.followup_dataset import load
from recommendation.explain import followup_explain as fx

table, _ = load(P["followup_dataset"], MODELS / "followup_preprocessor.json")
res = fx.run(table, joblib.load(MODELS / "followup_trajectory.joblib"), joblib.load(MODELS / "followup_risk.joblib"))
pd.DataFrame(res["checks"])

## 1. What drives the predictions (share of mean |SHAP|)

In [ ]:
imp = pd.DataFrame(res["global_importance"])
(imp / imp.sum() * 100).round(0)

## 2. Example test grafts: reasons and counterfactuals

In [ ]:
for e in res["examples"]:
    display(Markdown(f'**{e["graft"]}** ({e["procedure"]}, 12-month ECD {e["ECD_12m"]:.0f}): predicted 60-month ECD '
                     f'{e["predicted_ECD_60m"]} - {e["why_trajectory"]}'))
    for ep in ("graft_failure", "low_ecd"):
        d = e[ep]
        if "note" in d:
            print(f"   {ep}: {d['note']}")
        else:
            print(f"   {ep} risk {d['risk_24m']:.0%}: {d['why']}")
            if d["counterfactual"]:
                print(f"      {d['counterfactual']}")